In [ ]:
import os
import traceback
import numpy as np

# Setup the module path.
import sys
sys.path.append(r"C:\Users\leila\Documents\Visual Studio\pppl_xics_2026\xicsrt")
sys.path.append(r"C:\Users\leila\Documents\Visual Studio\pppl_xics_2026\xicsrt_contrib")
sys.path.append(r"C:\Users\leila\Documents\Visual Studio\pppl_xics_2026\xicsrt_analysis")
sys.path.append(r"C:\Users\leila\Documents\Visual Studio\pppl_xics_2026\mirfusion_library")
sys.path.append(r"C:\Users\leila\Documents\Visual Studio\pppl_xics_2026\mirxics_jax")
import xicsrt

from xicsrt.xicsrt_io import load_config

In [ ]:
"""
Load saved XICSRT configuration files, run the ray tracing, and save
the corresponding detector images.
"""

def generate_results_files(num_result_files=None, config_path=None, results_path=None, overwrite=False, stop_on_error=False):
    """
    Load saved XICSRT configuration files, run the ray tracing, and
    save the 'detector origin' from the results.

    Parameters
    ----------
    num_result_files : maximum number of configuration files to process. If None, then process every JSON configuration file in config_path.
    config_path : directory containing the saved XICSRT configuration files. 
    detector_path : directory in which the detector images are saved. 
    overwrite : bool, optional
        If True, overwrite detector image files that already exist.
        If False, skip detector image files that have already been saved.
    stop_on_error : bool, optional
        If True, stop when a ray trace fails. If False, record the
        error and continue processing the remaining configurations.
    """
    if config_path is None:
        raise ValueError("There is no config_path listed. Config files failed to load.")
        return

    if results_path is None:
        raise ValueError("There is no result_path listed. Result files failed to save.")
        return

    if not os.path.isdir(config_path):
        raise FileNotFoundError(f"Configuration directory does not exist:\n{config_path}")

    os.makedirs(detector_path, exist_ok=True)

    # Find all saved JSON configuration files.
    config_filenames = sorted(
        filename
        for filename in os.listdir(config_path)
        if filename.lower().endswith(".json")
    )

    if not config_filenames:
        raise FileNotFoundError(f"No JSON configuration files were found in:\n{config_path}")

    # Optionally process only a specified number of files.
    # If value is None, every config.json file in the config_path will be processed.
    if num_result_files is not None:
        config_filenames = config_filenames[:num_detector_images]

    error_log_path = os.path.join(detector_path, "raytrace_errors.txt")

    num_completed = 0
    num_skipped = 0
    num_failed = 0

    for file_number, config_filename in enumerate(config_filenames, start=1):

        config_filepath = os.path.join(config_path, config_filename)

        try:
            # Load the exact configuration that was previously saved.
            # This includes the random plasma profiles and all XICSRT settings.
            config = load_config(config_filepath)
            
            if config["sources"]["plasma"].get("wout_file") is None:
                config["sources"]["plasma"]["wout_file"] = (
                    r"C:\Users\leila\Documents\Visual Studio"
                    r"\pppl_xics_2026\wout.nc"
            )

            # Read the sample index stored inside the configuration.
            sample_index = config["scenario"]["sample_index"]

            print(
                f"Processing sample {sample_index:06d} "
                f"({file_number}/{len(config_filenames)})"
            )

            existing_detector_files = [
                filename
                for filename in os.listdir(detector_path)
                if filename.startswith(
                    f"xicsrt_detector_{sample_index:06d}"
                )
            ]

            if existing_detector_files and not overwrite: 
                num_skipped += 1
                print(
                    f"[{file_number}/{len(config_filenames)}] "
                    f"Skipping sample {sample_index:06d} "
                    f"({len(existing_detector_files)} detector file(s) already exist)"
                )
                continue

            print(
                f"[{file_number}/{len(config_filenames)}] "
                f"Loading {config_filename}"
            )

            # Give every sample a unique XICSRT output suffix
            output_suffix = f"{sample_index:06d}"

            # Saving results after each config file is processed
            config["general"]["save_images"] = False
            config["general"]["save_results"] = False
            config["general"]["keep_history"] = True
            config["general"]["results_ext"] = ".hdf5"
            config["general"]["output_path"] = results_path
            config["general"]["output_suffix"] = output_suffix
            config["general"]["make_directories"] = True

            print(
                f"Running ray trace for sample "
                f"{sample_index:06d}"
            )
            
            # Pass the loaded configuration directly into XICSRT.
            result = xicsrt.raytrace(config)

            # Extract the detector image from the ray-tracing result.
            detector_image = np.asarray(result["total"]["image"]["detector"])

            num_completed += 1

            print(f"Saved detector images for sample "
                 f"{sample_index:06d}"
            )
            print(f"Detector image shape: {detector_image.shape}")

        except Exception:
            num_failed += 1
            error_message = (
                "\n"
                "========================================\n"
                f"Configuration: {config_filepath}\n"
                f"{traceback.format_exc()}"
                "========================================\n"
            )
            print(f"Failed to process {config_filename}.")
            with open(error_log_path, "a", encoding="utf-8") as error_file:
                error_file.write(error_message)
            if stop_on_error:
                raise

    print("\nDetector image generation complete")
    print("----------------------------------")
    print(f"Completed: {num_completed}")
    print(f"Skipped:   {num_skipped}")
    print(f"Failed:    {num_failed}")

In [ ]:
config_path = r"C:\Users\leila\Documents\Visual Studio\pppl_xics_2026\xics_ml_pipeline\nn_training_data\xicsrt_profile_configs"
results_path = r"C:\Users\leila\Documents\Visual Studio\pppl_xics_2026\xics_ml_pipeline\nn_training_data\xicsrt_results_files"
generate_results_files(num_detector_images=2, config_path=config_path, results_path=detector_path, overwrite=False, stop_on_error=True)